In [ ]:
!pip install -q openpyxl pandas requests tqdm

In [ ]:
import os
import json
import time
import requests
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from google.colab import userdata


# ============================
# Configuration
# ============================

OPENROUTER_API_KEY = "your api key here"

MODEL = "openrouter/free"
# Change to any OpenRouter model if needed
# Examples:
# MODEL = "qwen/qwen3-8b:free"
# MODEL = "deepseek/deepseek-chat-v3:free"

INPUT_FILE = "mydata.xlsx"
OUTPUT_FILE = "cleaned_dataset.xlsx"

QUESTION_COLUMN = "Question"
ANSWER_COLUMN = "Answer"

# ============================
# Prompt
# ============================

SYSTEM_PROMPT = """
You are an expert dataset cleaning assistant.

Your ONLY task is to improve grammar, spelling, punctuation, capitalization,
and readability.

IMPORTANT RULES:

- DO NOT change the meaning.
- DO NOT add any new information.
- DO NOT remove information.
- Keep answers in first person if they already are.
- Do not make the language unnecessarily formal.
- Preserve technical terms exactly.
- Return ONLY valid JSON.
- No markdown.
- No explanations.

JSON format:

{
  "question": "...",
  "answer": "..."
}
"""

# ============================
# OpenRouter API
# ============================

URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

MAX_RETRIES = 5

def clean_qa(question, answer):

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Question:\n{question}\n\nAnswer:\n{answer}"
            }
        ],
        "temperature": 0
    }

    for attempt in range(MAX_RETRIES):

        response = requests.post(
            URL,
            headers=HEADERS,
            json=payload,
            timeout=120
        )

        # Handle rate limit
        if response.status_code == 429:

            wait = (attempt + 1) * 5

            print(f"Rate limited. Waiting {wait} seconds...")

            time.sleep(wait)

            continue

        response.raise_for_status()

        content = response.json()["choices"][0]["message"]["content"]

        content = content.strip()

        # Remove markdown
        if content.startswith("```"):

            content = content.replace("```json", "")
            content = content.replace("```", "")
            content = content.strip()

        # Extract JSON
        start = content.find("{")
        end = content.rfind("}")

        if start == -1 or end == -1:
            raise ValueError(f"No JSON found:\n{content}")

        content = content[start:end+1]

        cleaned = json.loads(content)

        return cleaned["question"], cleaned["answer"]

    raise Exception("Maximum retries exceeded.")


# ============================
# Main
# ============================

def main():

    if OPENROUTER_API_KEY is None:
        raise ValueError("OPENROUTER_API_KEY not found in .env")

    df = pd.read_excel(INPUT_FILE, header=None)

    cleaned_questions = []
    cleaned_answers = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Cleaning"):

        question = str(row[0]).strip()   # First column
        answer = str(row[1]).strip()     # Second column

        try:
            clean_q, clean_a = clean_qa(question, answer)

        except Exception as e:
            print(f"\nError: {e}")
            clean_q = question
            clean_a = answer

        cleaned_questions.append(clean_q)
        cleaned_answers.append(clean_a)

        time.sleep(1)

    output_df = pd.DataFrame({
        "Original Question": df[0],
        "Original Answer": df[1],
        "Clean Question": cleaned_questions,
        "Clean Answer": cleaned_answers
    })

    output_df.to_excel(OUTPUT_FILE, index=False)

    print(f"\nDone! Saved cleaned dataset to '{OUTPUT_FILE}'")


if __name__ == "__main__":
    main()

Cleaning:   8%|▊         | 9/116 [00:39<08:49,  4.95s/it]

Rate limited. Waiting 5 seconds...
Rate limited. Waiting 10 seconds...


Cleaning:   9%|▊         | 10/116 [01:00<17:32,  9.93s/it]

Rate limited. Waiting 5 seconds...


Cleaning:   9%|▉         | 11/116 [01:09<16:46,  9.58s/it]

...

Rate limited. Waiting 5 seconds...


Cleaning:  99%|█████████▉| 115/116 [18:16<00:05,  5.80s/it]

Rate limited. Waiting 5 seconds...


Cleaning: 100%|██████████| 116/116 [18:26<00:00,  9.54s/it]


Done! Saved cleaned dataset to 'cleaned_dataset.xlsx'


In [ ]:
import pandas as pd

df = pd.read_excel("/content/cleaned_dataset.xlsx")

# Count missing values in each column
print(df.isnull().sum())

# Total missing values in dataset
print("\nTotal missing values:", df.isnull().sum().sum())

Original Question    0
Original Answer      0
Clean Question       0
Clean Answer         0
dtype: int64

Total missing values: 0


In [ ]:
import pandas as pd

df = pd.read_excel("/content/cleaned_dataset.xlsx")

for col in df.columns:
    missing = (
        df[col].isna() |
        (df[col].astype(str).str.strip() == "")
    ).sum()

    print(f"{col}: {missing}")

Original Question: 0
Original Answer: 0
Clean Question: 0
Clean Answer: 0


In [ ]:
missing_rows = df[
    df.isna().any(axis=1)
]

print(f"Rows with missing values: {len(missing_rows)}")
print(missing_rows.head())

Rows with missing values: 1
            Original Question Original Answer                  Clean Question  \
115  whats your fav dc movie?             NaN  What's your favorite DC movie?   

    Clean Answer  
115          NaN  


In [ ]:
from sklearn.model_selection import train_test_split

# First separate test set
train_val_df, test_df = train_test_split(
    df,
    test_size=10,
    random_state=42,
    shuffle=True
)

# Then separate validation set
train_df, val_df = train_test_split(
    train_val_df,
    test_size=10,
    random_state=42,
    shuffle=True
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 95
Validation: 10
Test: 10


In [ ]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)